# 02a -- 02a_fact_nfl_combine_pro_day_metrics (All Seasons)

**Purpose**: Single combine/pro-day table covering all nflverse seasons.
Loads once, normalizes column names, left-joins `dim_nfl_players` for `gsis_id`
and `dim_rookie_prospect` for the interim `player_key`, then flags the current
draft class with `is_current_season`.

**Key**: `pfr_id + season`

**`gsis_id`**: FK to `dim_nfl_players` (null for current rookies pre-signing).

**`player_key`**: interim FK to `dim_rookie_prospect` (null for historical seasons).

**`is_current_season`**: `True` when `season == CFG.draft_year`.

**Prerequisites**:
- `01a_dim_rookie_prospect.ipynb` must have run (`dim_rookie_prospect.parquet`)
- `01e_dim_nfl_players_seed.ipynb` must have run (`dim_nfl_players.parquet`)

**Outputs**:
- `data/fact_nfl_combine_pro_day_metrics.parquet`

In [14]:
import pandas as pd
import numpy as np
import nflreadpy as nfl
from pathlib import Path
from datetime import datetime, date
from dataclasses import dataclass, field


@dataclass
class LeagueConfig:
    draft_year: int = 2026
    total_cap: int = 500_000_000
    num_teams: int = 28
    num_conferences: int = 2
    initial_contract_years: int = 3
    extension_contract_years: int = 3
    fa_minimum_salary: int = 2_000_000
    data_dir: str = "data"
    fuzzy_auto_threshold: int = 90
    fuzzy_review_threshold: int = 70


CFG = LeagueConfig()
DATA = Path(CFG.data_dir)
TODAY = date.today().isoformat()
DATA.mkdir(exist_ok=True)

## 1 -- Helper: parse_height_to_inches

Copied from `01a_dim_rookie_prospect` so this notebook runs standalone.

In [15]:
# parse_height_to_inches imported from etl_helpers (was a standalone copy of 01's).
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
from etl_helpers import parse_height_to_inches

parse_height_to_inches: all smoke tests passed


## 2 -- Load All-Season Combine Data

In [16]:
# -- Load all seasons from nflreadpy ─────────────────────────────────────────
print("Loading all-season combine data from nflverse...")
try:
    combine_raw = nfl.load_combine(seasons=True).to_pandas()
except Exception as e:
    raise RuntimeError(f"Failed to load nflverse combine data: {e}") from e

print(f"Total rows    : {len(combine_raw):,}")
print(f"Season range  : {combine_raw['season'].min()} - {combine_raw['season'].max()}")
print(f"Columns ({len(combine_raw.columns)}): {list(combine_raw.columns)}")

Loading all-season combine data from nflverse...
Total rows    : 8,968
Season range  : 2000 - 2026
Columns (18): ['season', 'draft_year', 'draft_team', 'draft_round', 'draft_ovr', 'pfr_id', 'cfb_id', 'player_name', 'pos', 'school', 'ht', 'wt', 'forty', 'bench', 'vertical', 'broad_jump', 'cone', 'shuttle']


## 3 -- Build fact_nfl_combine_pro_day_metrics

Single table: normalized metric columns + identity columns from nflverse +
`player_key` (left join) + `is_current_season` flag.

In [17]:
# -- Column maps ─────────────────────────────────────────────────────────────
_COL_MAP = {
    "ht":         "_ht_raw",
    "wt":         "weight",
    "forty":      "forty_yard",
    "bench":      "bench_press",
    "vertical":   "vertical_jump",
    "broad_jump": "broad_jump",
    "cone":       "three_cone",
    "shuttle":    "shuttle",
}
_OPTIONAL_METRICS = {
    "ten_split":  "ten_split",
    "hand_size":  "hand_size",
    "arm_length": "arm_length",
    "wingspan":   "wingspan",
}
_IDENTITY_COLS = [
    "pfr_id", "cfb_id", "season",
    "player_name", "pos", "school",
    "draft_team", "draft_round", "draft_ovr",
]

# gsis_id is the long-term FK to dim_nfl_players (null pre-signing for current class)
# player_key is the interim FK from dim_rookie_prospect (null for non-prospect seasons)
_SCHEMA = [
    # Composite key
    "pfr_id", "season",
    # Player registry links
    "gsis_id",            # FK -> dim_nfl_players.gsis_id (null pre-signing)
    "player_key",         # Interim FK -> dim_rookie_prospect.player_key (null for non-prospects)
    "is_current_season",
    # Identity
    "cfb_id", "player_name", "pos", "school",
    "draft_team", "draft_round", "draft_ovr",
    "metric_source",
    # Physical
    "height_inches", "weight",
    # Athletic drills
    "forty_yard", "ten_split",
    "bench_press", "vertical_jump", "broad_jump",
    "three_cone", "shuttle",
    # Extended (optional)
    "hand_size", "arm_length", "wingspan",
]


def build_fact_nfl_combine_pro_day_metrics(combine_df: pd.DataFrame, cfg: LeagueConfig) -> pd.DataFrame:
    for col in ["pfr_id", "season"]:
        if col not in combine_df.columns:
            raise ValueError(f"Required column '{col}' missing from nflverse combine data")

    # Drop rows missing the composite key
    before = len(combine_df)
    df = combine_df[
        combine_df["pfr_id"].notna() & combine_df["season"].notna()
    ].copy()
    if before != len(df):
        print(f"  Dropped {before - len(df)} rows missing pfr_id or season")

    df = df.drop_duplicates(subset=["pfr_id", "season"], keep="last").reset_index(drop=True)

    # Rename metric columns
    col_map = {
        **_COL_MAP,
        **{k: v for k, v in _OPTIONAL_METRICS.items() if k in df.columns},
    }
    df = df.rename(columns=col_map)

    if "_ht_raw" in df.columns:
        df["height_inches"] = df["_ht_raw"].apply(parse_height_to_inches)
        df.drop(columns=["_ht_raw"], inplace=True)

    df["is_current_season"] = df["season"] == cfg.draft_year

    # -- Join gsis_id from dim_nfl_players (central registry) via pfr_id -----
    # Left join: populated for historical players; null for current rookies pre-signing
    nfl_players = pd.read_parquet(DATA / "dim_nfl_players.parquet")
    gsis_lookup = (
        nfl_players[["gsis_id", "pfr_id"]]
        .dropna(subset=["pfr_id"])
        .drop_duplicates(subset=["pfr_id"])
    )
    df = df.merge(gsis_lookup, on="pfr_id", how="left")

    # -- Join player_key from dim_rookie_prospect (interim FK) via pfr_id ----
    # Left join: populated for current draft class; null for historical seasons
    prospects = pd.read_parquet(DATA / "dim_rookie_prospect.parquet")
    key_lookup = (
        prospects[["player_key", "pfr_id"]]
        .dropna(subset=["pfr_id"])
        .drop_duplicates(subset=["pfr_id"])
    )
    df = df.merge(key_lookup, on="pfr_id", how="left")

    df["metric_source"] = "combine"

    for col in _SCHEMA:
        if col not in df.columns:
            df[col] = pd.NA

    return df[_SCHEMA].reset_index(drop=True)


print("Building fact_nfl_combine_pro_day_metrics...")
combine_metrics_06 = build_fact_nfl_combine_pro_day_metrics(combine_raw, CFG)

out_path = DATA / "fact_nfl_combine_pro_day_metrics.parquet"
combine_metrics_06.to_parquet(out_path, index=False)

curr = combine_metrics_06[combine_metrics_06["is_current_season"]]
hist = combine_metrics_06[~combine_metrics_06["is_current_season"]]
print(f"Total rows            : {len(combine_metrics_06):,}")
print(f"Current season ({CFG.draft_year}) : {len(curr)} rows")
print(f"  gsis_id populated   : {curr['gsis_id'].notna().sum()} (expected low pre-signing)")
print(f"  player_key populated: {curr['player_key'].notna().sum()}")
print(f"Historical seasons    : {len(hist):,} rows")
print(f"  gsis_id populated   : {hist['gsis_id'].notna().sum()}")
print(f"Saved -> {out_path}")
curr.head(5)

Building fact_nfl_combine_pro_day_metrics...
  Dropped 1531 rows missing pfr_id or season
Total rows            : 7,434
Current season (2026) : 265 rows
  gsis_id populated   : 220 (expected low pre-signing)
  player_key populated: 265
Historical seasons    : 7,169 rows
  gsis_id populated   : 6828
Saved -> data\fact_nfl_combine_pro_day_metrics.parquet


,pfr_id,season,gsis_id,player_key,is_current_season,cfb_id,player_name,pos,school,draft_team,...,forty_yard,ten_split,bench_press,vertical_jump,broad_jump,three_cone,shuttle,hand_size,arm_length,wingspan
7169,AdamCh01,2026,NaN,4f61e36da0f6,True,chris-adams-2,Chris Adams,OT,Memphis,NaN,...,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>
7170,AllaDr00,2026,ALL015451,e5417bc86706,True,drew-allar-1,Drew Allar,QB,Penn St.,NaN,...,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>
7171,AlleKa01,2026,ALL532376,85fc4414cd8f,True,kaytron-allen-1,Kaytron Allen,RB,Penn St.,NaN,...,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>
7172,AlleMa05,2026,ALL595143,3df670e6c8ad,True,marcus-allen-6,Marcus Allen,CB,North Carolina,NaN,...,4.50,<NA>,NaN,39.0,123.0,NaN,NaN,<NA>,<NA>,<NA>
7173,AltmLu00,2026,NaN,0fb64b35d726,True,luke-altmyer-1,Luke Altmyer,QB,Illinois,NaN,...,4.72,<NA>,NaN,32.0,114.0,NaN,NaN,<NA>,<NA>,<NA>


## 4 -- Validation

In [18]:
df      = pd.read_parquet(DATA / "fact_nfl_combine_pro_day_metrics.parquet")
dim_nfl = pd.read_parquet(DATA / "dim_nfl_players.parquet")
dim_rp  = pd.read_parquet(DATA / "dim_rookie_prospect.parquet")

print(f"fact_nfl_combine_pro_day_metrics: {len(df):,} rows, {len(df.columns)} columns")
print(f"Composite key unique: {not df.duplicated(subset=['pfr_id', 'season']).any()}")
print()

curr = df[df["is_current_season"]]
hist = df[~df["is_current_season"]]
print(f"is_current_season=True  : {len(curr):>6} rows")
print(f"is_current_season=False : {len(hist):>6} rows")
print()

print("gsis_id (dim_nfl_players FK) population:")
print(f"  Current season : {curr['gsis_id'].notna().sum()}/{len(curr)} (expected low pre-signing)")
print(f"  Historical     : {hist['gsis_id'].notna().sum()}/{len(hist)}")
print()

print("player_key (dim_rookie_prospect FK) population:")
print(f"  Current season : {curr['player_key'].notna().sum()}/{len(curr)}")
print(f"  Historical     : {hist['player_key'].notna().sum()}/{len(hist)} (expected 0)")
print()

print(f"Drill coverage for {CFG.draft_year} invitees:")
drill_cols = ["forty_yard", "bench_press", "vertical_jump", "broad_jump", "three_cone", "shuttle"]
for col in drill_cols:
    n   = curr[col].notna().sum()
    pct = n / len(curr) if len(curr) else 0
    print(f"  {col:<18} {pct:.1%}  ({n}/{len(curr)})")
print()

nfl_gsis    = set(dim_nfl["gsis_id"].dropna())
rp_keys     = set(dim_rp["player_key"].dropna())
orphan_gsis = set(df["gsis_id"].dropna()) - nfl_gsis
orphan_keys = set(df["player_key"].dropna()) - rp_keys
print(f"gsis_id integrity   : {'OK' if not orphan_gsis else f'WARN {len(orphan_gsis)} orphaned'}")
print(f"player_key integrity: {'OK' if not orphan_keys else f'WARN {len(orphan_keys)} orphaned'}")

fact_nfl_combine_pro_day_metrics: 7,434 rows, 25 columns
Composite key unique: True

is_current_season=True  :    265 rows
is_current_season=False :   7169 rows

gsis_id (dim_nfl_players FK) population:
  Current season : 220/265 (expected low pre-signing)
  Historical     : 6828/7169

player_key (dim_rookie_prospect FK) population:
  Current season : 265/265
  Historical     : 0/7169 (expected 0)

Drill coverage for 2026 invitees:
  forty_yard         58.5%  (155/265)
  bench_press        15.1%  (40/265)
  vertical_jump      58.5%  (155/265)
  broad_jump         54.0%  (143/265)
  three_cone         15.1%  (40/265)
  shuttle            17.4%  (46/265)

gsis_id integrity   : OK
player_key integrity: OK
